# Task 2: Fashion Season Classification

**COSC2753 Machine Learning, Assignment 2 (2026B)**

## Executive Summary

- **Problem.** Predict `season` from each catalogue image.
- **Evaluation first.** Section 2 fixes the split, metrics and simple baselines before any model is trained.
- **Primary metric.** Macro-F1 gives every season equal importance. Accuracy, balanced accuracy, weighted F1 and top-2 accuracy provide supporting views.
- **Three candidate models.** A Random Forest over engineered visual features, EfficientNet-B0, and DenseNet-121 compare classical learning, efficient convolution and dense feature reuse.
- **Fair comparison.** Every candidate uses the same training and validation rows. The two neural networks share the same training loop and stopping rule.
- **Final judgement.** The selected model is judged using per-season results, confusion patterns, stability, calibration and deployment cost.

All data preparation is inherited from `00_eda_and_preprocessing.ipynb`. Run that notebook first.


## How to Run

Run the notebook from top to bottom after `00_eda_and_preprocessing.ipynb` has produced `preprocessed_datasets/train_manifest.csv`.

Install the required packages:

```text
pip install torch torchvision numpy pandas scikit-learn scikit-image matplotlib seaborn tqdm joblib
```

The full neural runs require a CUDA GPU. Set `QUICK_RUN = True` for a short structural check. Finished models and interrupted neural runs are stored under `models/task2/checkpoints/` when `RESUME = True`.

Main outputs:

- `models/task2/task2_results.csv`
- `models/task2/task2_model.pt` or `task2_random_forest.joblib`
- `models/task2/task2_classes.json`
- `outputs/task2_predictions.csv`


## 1. Setup


In [ ]:
%matplotlib inline

import gc
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, densenet121

from joblib import Parallel, delayed, dump as joblib_dump, load as joblib_load
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0"]
MUTED = "#6b7280"


In [ ]:
# The shared data access module written for notebook 00. Importing it rather than
# redefining the transform is what keeps all four task notebooks reading identical pixels.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

try:
    from src.preprocessing import (
        MANIFEST,
        TEST_IMAGE_DIR,
        IMAGE_TARGET_SIZE,
        compute_normalisation,
        describe_split,
        load_image_array,
        load_manifest,
        make_split,
    )
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Could not import src.preprocessing from {REPO_ROOT}. "
        "This notebook must sit in notebooks/ inside the repository root."
    ) from error

if not MANIFEST.exists():
    raise FileNotFoundError(
        f"{MANIFEST} is missing. Run 00_eda_and_preprocessing.ipynb first; "
        "Section 3.4 of that notebook writes it."
    )

print("Repository root:", REPO_ROOT)
print("Manifest:", MANIFEST)
print("Image target size (w, h):", IMAGE_TARGET_SIZE)

### 1.1 Configuration

All settings used later are defined here. The two neural networks share the same optimisation budget. `USE_PRETRAINED` is kept `False` so both networks are trained from scratch; change it only if pretrained models are permitted, and apply the same policy to both models.


In [ ]:
TARGET = "season"
RANDOM_STATE = 42

QUICK_RUN = True
RUN_SALIENCY = True
RESUME = False
CHECKPOINT_EVERY = 1
KEEP_EPOCH_CHECKPOINTS = False

ALLOW_CPU = False
USE_AMP = True
CHANNELS_LAST = True
CACHE_ON_DEVICE = True
USE_COMPILE = False
DETERMINISTIC = False
FEATURE_N_JOBS = -1

VALIDATION_SHARE = 0.20
BATCH_SIZE = 128
EPOCHS = 30
PATIENCE = 6
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
LABEL_SMOOTHING = 0.05

AUG_FLIP_PROBABILITY = 0.5
AUG_ROTATION_DEGREES = 8.0
AUG_TRANSLATE_FRACTION = 0.06
AUG_JITTER_STRENGTH = 0.05  # mild: colour may be useful for season

RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 2
RF_MAX_FEATURES = "sqrt"
USE_PRETRAINED = False

ARTEFACT_DIR = REPO_ROOT / "models" / "task2"
CHECKPOINT_DIR = ARTEFACT_DIR / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs"
PREDICTION_TEMPLATE = REPO_ROOT / "datasets" / "test" / "styles_prediction.csv"

if QUICK_RUN:
    EPOCHS, WARMUP_EPOCHS = 3, 1
    ALLOW_CPU = True
    RF_N_ESTIMATORS = 100

for directory in (ARTEFACT_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("QUICK_RUN:", QUICK_RUN, "| epochs:", EPOCHS)
print("RESUME:", RESUME, "| checkpoints ->", CHECKPOINT_DIR)


#### Performance and recovery notes

- `QUICK_RUN` checks the complete workflow with fewer rows and epochs; its metrics are not reportable.
- `RESUME` restores compatible saved models and full neural training states.
- `USE_AMP`, channels-last tensors and device caching improve GPU throughput.
- A configuration fingerprint prevents checkpoints from a different experiment being reused silently.


In [ ]:
def set_seed(seed):
    """Seed every generator this notebook draws from.

    Torch, NumPy and Python are all seeded because the augmentation, the weight
    initialisation and the batch order each draw from a different one. Without all three,
    a "same seed" rerun is not actually a rerun.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def capture_rng_state():
    """Every generator's state, so a resumed run continues the same random stream.

    Without this, resuming at epoch 20 would draw a different augmentation sequence from the
    one an uninterrupted run would have drawn, and the resumed run would not be the same run.
    """
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    """Reinstate the generators captured by `capture_rng_state`."""
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu() if torch.is_tensor(state["torch"])
                        else state["torch"])
    if "cuda" in state and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if DEVICE.type == "cuda":
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {name} | {total:.1f} GB | compute capability {capability[0]}.{capability[1]}")
    print(f"torch {torch.__version__} built against CUDA {torch.version.cuda}")

    # A wheel without kernels for this card fails here rather than mid-epoch, and the failure
    # is a clear one instead of a process that disappears without a traceback.
    try:
        _ = (torch.zeros(8, device=DEVICE) + 1).sum().item()
    except RuntimeError as error:
        raise RuntimeError(
            f"CUDA is visible but cannot execute a kernel on this card: {error}\n"
            "This is the usual symptom of a PyTorch build without kernels for your GPU. "
            "RTX 50-series cards need a CUDA 12.8 build:\n"
            "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128"
        ) from error

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif DEVICE.type == "cpu" and not ALLOW_CPU:
    raise RuntimeError(
        "No supported GPU (CUDA or Apple MPS) is available. Training these neural networks on a CPU takes roughly 10 minutes per epoch "
        "per thousand images, which looks like a hang rather than an error.\n"
        "Either install a CUDA build of PyTorch, or set ALLOW_CPU = True in Section 1.1 "
        "and QUICK_RUN = True to accept the slowdown deliberately."
    )
elif DEVICE.type == "cpu":
    print("Running on CPU by explicit request (ALLOW_CPU = True). Expect this to be slow.")
else:
    print("Apple Metal GPU detected; using the PyTorch MPS backend.")

# Determinism is a deliberate trade rather than a default. cudnn.benchmark autotunes the
# convolution algorithm for this fixed input size and is worth real throughput; turning it off
# together with deterministic kernels is what makes two same-seed runs bit-identical.
if DETERMINISTIC:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except (AttributeError, RuntimeError) as error:
        print("Deterministic algorithms unavailable:", error)
else:
    torch.backends.cudnn.benchmark = DEVICE.type == "cuda"

# bf16 has the dynamic range of fp32 and needs no loss scaling. Turing cards such as the T4
# lack it, so fp16 with a gradient scaler is the fallback. Neither changes what is learned.
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
AMP_DTYPE = (torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16)
CHANNELS_LAST = CHANNELS_LAST and DEVICE.type == "cuda"
CACHE_ON_DEVICE = CACHE_ON_DEVICE and DEVICE.type == "cuda"
USE_COMPILE = USE_COMPILE and hasattr(torch, "compile")

print(f"Mixed precision: {AMP_ENABLED} ({str(AMP_DTYPE).replace('torch.', '') if AMP_ENABLED else 'fp32'})"
      f" | channels_last: {CHANNELS_LAST} | image cache on device: {CACHE_ON_DEVICE}")
print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark} | deterministic: {DETERMINISTIC} "
      f"| torch.compile: {USE_COMPILE}")

set_seed(RANDOM_STATE)
print("Seeded at", RANDOM_STATE)

## 2. Data and the Evaluation Framework

This section is fixed before model training. Every candidate is evaluated on the same validation rows with the same metrics.


### 2.1 Leakage-safe split

The shared `make_split` function keeps identical-image groups on one side of the split and stratifies by `season`. Classes with too few independent groups remain in training. The class mapping is fitted from training labels and saved with the final model.


In [ ]:
frame = load_manifest(TARGET)
print(f"Rows carrying an {TARGET} label: {len(frame):,}")
print(f"Distinct classes in the manifest: {frame[TARGET].nunique()}")

train_frame, val_frame = make_split(
    frame, TARGET, validation_share=VALIDATION_SHARE, random_state=RANDOM_STATE
)

if QUICK_RUN:
    # Stratified where possible; a plain sample is enough for a structural check.
    train_frame = train_frame.sample(n=min(5000, len(train_frame)), random_state=RANDOM_STATE)
    train_frame = train_frame.reset_index(drop=True)
    print("QUICK_RUN: training rows reduced to", len(train_frame))

display(describe_split(train_frame, val_frame, TARGET))

In [ ]:
# Label encoding. Fixed to the sorted training classes and persisted in Section 8, because
# reconstructing it later from a different frame would silently permute every prediction.
CLASSES = sorted(train_frame[TARGET].unique())

if QUICK_RUN:
    # The subsample above can drop whole classes, which would leave validation rows with no
    # index to map to. Full runs never enter this branch: make_split keeps every class in training.
    keep = val_frame[TARGET].isin(CLASSES)
    print(f"QUICK_RUN: dropping {int((~keep).sum())} validation rows whose class was "
          "removed by the training subsample.")
    val_frame = val_frame.loc[keep].reset_index(drop=True)

CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)

# Any validation class absent from training cannot be predicted. make_split sends
# single-group classes to training, so this should be empty; the check is what proves it.
unseen = sorted(set(val_frame[TARGET]) - set(CLASSES))
assert not unseen, f"Validation holds classes never seen in training: {unseen}"

y_train = train_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()
y_val = val_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()

train_support = pd.Series(np.bincount(y_train, minlength=N_CLASSES), index=CLASSES)
val_support = pd.Series(np.bincount(y_val, minlength=N_CLASSES), index=CLASSES)
SCOREABLE = np.flatnonzero(val_support.to_numpy() > 0)   # class indices macro averages use

print(f"Classes: {N_CLASSES}")
print(f"Scoreable in validation: {len(SCOREABLE)} | absent: {N_CLASSES - len(SCOREABLE)}")
print(f"Training support range: {train_support.max():,} down to {train_support.min()}")
print("Absent from validation:", sorted(np.array(CLASSES)[val_support.to_numpy() == 0]))

In [ ]:
class_table = pd.DataFrame({
    "Training images": train_support,
    "Validation images": val_support,
})
class_table["Training share %"] = class_table["Training images"] / len(y_train) * 100
display(class_table.style.format({"Training share %": "{:.1f}%"}))


### 2.2 Loading images into memory

The images are decoded once using the deterministic transform from notebook 00 and retained as `uint8`. Training augmentation is still sampled separately for every batch.


In [ ]:
def build_image_cache(frame, description):
    """Decode a frame's images once through the shared transform into one uint8 array.

    Only the deterministic transform from Section 3.1 of notebook 00 is applied here, exactly
    once per image. Augmentation still happens per epoch, so nothing about the training
    distribution is frozen by this cache.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8, in the frame's row order.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


def report_memory(label=""):
    """Host RSS and, on CUDA, device allocation. Cheap, and it makes a leak visible early."""
    line = []
    try:
        import resource
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        line.append(f"host peak {peak_kb / 1e6:.2f} GB")
    except (ImportError, AttributeError):
        try:
            import psutil
            line.append(f"host RSS {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        except ImportError:
            pass
    if DEVICE.type == "cuda":
        line.append(f"device allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB "
                    f"reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"[memory{' ' + label if label else ''}] " + " | ".join(line))


X_train_images = build_image_cache(train_frame, "train")
X_val_images = build_image_cache(val_frame, "validation")

assert len(X_train_images) == len(y_train) and len(X_val_images) == len(y_val)
report_memory("after caching")

### 2.3 Training-only normalisation

RGB mean and standard deviation are fitted on the Task 2 training rows only. The same constants are then applied to validation and test images.


In [ ]:
VERIFY_NORMALISATION = False   # True: re-decode from disk and assert the constants match

start = time.time()
# Sums are taken over the raw 0-255 values in float32 and divided by 255 at the end, which is
# the same statistic as scaling first: sum(x/255) is sum(x)/255, and the same for the squares.
# The reductions accumulate into float64, so the running totals stay exact at this scale while
# the working chunk stays float32. Promoting the chunk itself to float64 would cost four bytes
# per channel per pixel twice over, once for the chunk and once for its square.
total = np.zeros(3, dtype=np.float64)
total_square = np.zeros(3, dtype=np.float64)
n_pixels = 0
for begin in range(0, len(X_train_images), 2048):
    chunk = X_train_images[begin:begin + 2048].astype(np.float32)
    total += chunk.sum(axis=(0, 1, 2), dtype=np.float64)
    total_square += np.einsum("nhwc,nhwc->c", chunk, chunk, dtype=np.float64)
    n_pixels += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]

mean_raw = total / n_pixels
variance_raw = np.maximum(total_square / n_pixels - mean_raw ** 2, 0.0)
NORM_MEAN = (mean_raw / 255.0).astype(np.float32)
NORM_STD = np.maximum(np.sqrt(variance_raw) / 255.0, 1e-6).astype(np.float32)

print(f"Fitted on {len(train_frame):,} training rows in {time.time() - start:.1f}s "
      "(from the cache, no second decode pass)")
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

if VERIFY_NORMALISATION:
    reference_mean, reference_std = compute_normalisation(train_frame,
                                                          target_size=IMAGE_TARGET_SIZE)
    print("Reference mean:", np.round(reference_mean, 4))
    print("Reference std: ", np.round(reference_std, 4))
    assert np.allclose(NORM_MEAN, reference_mean, atol=1e-4), "Mean disagrees with notebook 00"
    assert np.allclose(NORM_STD, reference_std, atol=1e-4), "Std disagrees with notebook 00"
    print("Verified against compute_normalisation.")

# White studio backgrounds dominate, so a mean near 0.9 is the expected result rather than a bug.
assert (NORM_MEAN > 0.5).all(), "Unexpectedly dark mean; check the transform before continuing."

### 2.4 Augmentation and batching

Horizontal flips and mild geometric transforms preserve the product while adding framing variation. Vertical flips and random crops are excluded. Colour jitter is deliberately mild because colour may contain real season information. Its effect should be checked during tuning.


In [ ]:
LUMA = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)
NORM_MEAN_T = torch.tensor(NORM_MEAN, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
NORM_STD_T = torch.tensor(NORM_STD, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)


def augment_batch(x):
    """Apply the Section 2.4 policy to a batch, with independent parameters per sample.

    Args:
        x: float tensor of shape (n, 3, height, width) with values in [0, 1].

    Returns:
        A tensor of the same shape and range.
    """
    n = x.shape[0]
    device = x.device

    # Horizontal flip. torch.where selects per sample, so the draw is genuinely per image.
    flip = torch.rand(n, device=device) < AUG_FLIP_PROBABILITY
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)

    # Rotation and translation in one affine warp. The height/width factors correct for
    # affine_grid's normalised coordinates, without which the rotation would shear.
    height, width = x.shape[-2], x.shape[-1]
    angle = (torch.rand(n, device=device) * 2 - 1) * math.radians(AUG_ROTATION_DEGREES)
    shift_x = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    shift_y = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    cos, sin = torch.cos(angle), torch.sin(angle)

    theta = torch.zeros(n, 2, 3, device=device)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin * height / width
    theta[:, 0, 2] = shift_x
    theta[:, 1, 0] = sin * width / height
    theta[:, 1, 1] = cos
    theta[:, 1, 2] = shift_y

    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    # Offset by -1 so that grid_sample's zero padding lands on white once shifted back.
    x = F.grid_sample(x - 1.0, grid, mode="bilinear", padding_mode="zeros",
                      align_corners=False) + 1.0

    # Colour jitter: brightness, saturation, contrast, each with its own per-sample factor.
    def factor():
        return 1.0 + (torch.rand(n, 1, 1, 1, device=device) * 2 - 1) * AUG_JITTER_STRENGTH

    x = x * factor()
    grey = (x * LUMA).sum(dim=1, keepdim=True)
    x = (x - grey) * factor() + grey
    mean = grey.mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * factor() + mean

    return x.clamp_(0.0, 1.0)


class BatchStream:
    """Device-resident batches, replacing Dataset plus DataLoader.

    The images are held once as uint8 in the device's memory and converted to float per batch,
    so nothing is copied from the host during training and the host holds no per-batch buffers.

    Args:
        images: uint8 array of shape (rows, height, width, 3), or an existing device tensor
            to share with another stream.
        labels: integer class indices aligned to `images`.
        batch_size: rows per batch.
        augment: apply the Section 2.4 policy. Training only.
        shuffle: reorder each epoch. Ignored when `weights` is given.
        weights: per-row sampling weights for class-balanced draws with replacement. Used by
            the decoupled stage in Section 6.2.
    """

    def __init__(self, images, labels, batch_size=None, augment=False, shuffle=False,
                 weights=None):
        if torch.is_tensor(images):
            self.images = images                      # shared with another stream, not copied
        else:
            self.images = torch.from_numpy(np.ascontiguousarray(images))
            if CACHE_ON_DEVICE:
                self.images = self.images.to(DEVICE, non_blocking=True)
        # Labels may arrive as an array or as an existing device tensor from `variant`,
        # which avoids a pointless round trip through host memory on every derived stream.
        self.labels = (labels.to(device=DEVICE, dtype=torch.long) if torch.is_tensor(labels)
                       else torch.as_tensor(np.asarray(labels), dtype=torch.long,
                                            device=DEVICE))
        self.batch_size = batch_size or BATCH_SIZE
        self.augment = augment
        self.shuffle = shuffle
        if weights is None:
            self.weights = None
        elif torch.is_tensor(weights):
            self.weights = weights.to(device=self.images.device, dtype=torch.double)
        else:
            self.weights = torch.as_tensor(np.asarray(weights), dtype=torch.double,
                                           device=self.images.device)

    def variant(self, **overrides):
        """A stream over the same device tensor with different batching or sampling."""
        settings = {"batch_size": self.batch_size, "augment": self.augment,
                    "shuffle": self.shuffle, "weights": self.weights}
        settings.update(overrides)
        return BatchStream(self.images, self.labels, **settings)

    def __len__(self):
        return math.ceil(len(self.labels) / self.batch_size)

    def _order(self):
        n = len(self.labels)
        if self.weights is not None:
            return torch.multinomial(self.weights, n, replacement=True)
        if self.shuffle:
            return torch.randperm(n, device=self.images.device)
        return torch.arange(n, device=self.images.device)

    def __iter__(self):
        order = self._order()
        for start in range(0, len(order), self.batch_size):
            index = order[start:start + self.batch_size]
            batch = self.images[index].to(DEVICE, non_blocking=True)
            x = batch.permute(0, 3, 1, 2).float().div_(255.0)
            if self.augment:
                x = augment_batch(x)
            x = (x - NORM_MEAN_T) / NORM_STD_T
            if CHANNELS_LAST:
                x = x.contiguous(memory_format=torch.channels_last)
            yield x, self.labels[index.to(self.labels.device)]


def make_loaders(batch_size=BATCH_SIZE, weights=None):
    """Training and validation streams. Reuses the uploaded tensors; nothing is re-copied."""
    return (train_loader.variant(batch_size=batch_size, weights=weights,
                                 shuffle=weights is None),
            val_loader)


train_loader = BatchStream(X_train_images, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_loader = BatchStream(X_val_images, y_val, 512, augment=False, shuffle=False)

batch_images, batch_labels = next(iter(train_loader))
print("Batch tensor:", tuple(batch_images.shape), batch_images.dtype, batch_images.device)
print(f"Normalised range: [{batch_images.min():.2f}, {batch_images.max():.2f}]")
print(f"Batches per epoch: {len(train_loader)}")
report_memory("after upload")

In [ ]:
# Look at what the model actually receives. An augmentation bug is far cheaper to catch here
# than to diagnose from a training curve.
def to_displayable(x01):
    """A [0, 1] CHW tensor as an HWC array ready for imshow."""
    return x01.detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy()


sample_positions = np.random.RandomState(RANDOM_STATE).choice(len(y_train), 6, replace=False)
index = torch.as_tensor(sample_positions, device=train_loader.images.device)
raw = train_loader.images[index].to(DEVICE).permute(0, 3, 1, 2).float().div(255.0)

set_seed(RANDOM_STATE)
augmented = augment_batch(raw.clone())

fig, axes = plt.subplots(2, 6, figsize=(11, 4.4))
for column in range(6):
    axes[0, column].imshow(to_displayable(raw[column]))
    axes[0, column].set_title(CLASSES[y_train[sample_positions[column]]], fontsize=7)
    axes[1, column].imshow(to_displayable(augmented[column]))
    for row in range(2):
        axes[row, column].axis("off")
fig.suptitle("Deterministic transform (top) and one augmented draw (bottom)", y=1.02)
plt.tight_layout(); plt.show()

# White padding must survive the warp: an augmented border darker than the studio background
# would be a fill colour bug, and would teach the model a border cue that the test set lacks.
corners = augmented[:, :, :2, :2]
print(f"Corner pixels after augmentation: min {corners.min():.3f}, mean {corners.mean():.3f} "
      "(1.0 is white)")

### 2.5 Metrics

Macro-F1 is the primary metric because every season should contribute equally. Accuracy, balanced accuracy and weighted F1 provide complementary views. Top-2 accuracy measures whether the correct season appears among two suggestions; top-5 is not meaningful for this small label space.


In [ ]:
def evaluate_predictions(y_true, y_pred, scores=None, name=""):
    labels = SCOREABLE
    row = {
        "Model": name,
        "Top-1 accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Balanced accuracy": recall_score(y_true, y_pred, labels=labels,
                                             average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if scores is not None:
        k = min(2, scores.shape[1])
        top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
        row["Top-2 accuracy"] = np.mean([truth in choices for truth, choices in zip(y_true, top_k)])
    else:
        row["Top-2 accuracy"] = np.nan
    return pd.DataFrame([row])


def per_class_table(y_true, y_pred):
    rows = []
    for index in SCOREABLE:
        true_binary = y_true == index
        pred_binary = y_pred == index
        rows.append({
            "Season": CLASSES[index],
            "Support": int(true_binary.sum()),
            "Precision": (true_binary & pred_binary).sum() / max(pred_binary.sum(), 1),
            "Recall": (true_binary & pred_binary).sum() / max(true_binary.sum(), 1),
            "F1": f1_score(true_binary, pred_binary, zero_division=0),
        })
    return pd.DataFrame(rows)


RESULTS = []

def record(result_frame):
    RESULTS.append(result_frame)
    display(result_frame.style.format({c: "{:.4f}" for c in result_frame.columns if c != "Model"}))
    return result_frame


### 2.6 Simple baselines

The majority baseline measures what class frequency alone can achieve. The stratified-random baseline samples from the training prior and provides a second non-learning reference. Random Forest is treated as a candidate model in Section 3, not as a trivial baseline.


In [ ]:
majority_index = int(np.bincount(y_train, minlength=N_CLASSES).argmax())
majority_pred = np.full_like(y_val, majority_index)
majority_scores = np.zeros((len(y_val), N_CLASSES))
majority_scores[:, majority_index] = 1.0
print(f"Majority class: {CLASSES[majority_index]} "
      f"({train_support.iloc[majority_index]:,} training images)")
record(evaluate_predictions(y_val, majority_pred, majority_scores, "Baseline: majority class"))

prior = np.bincount(y_train, minlength=N_CLASSES) / len(y_train)
rng = np.random.RandomState(RANDOM_STATE)
stratified_pred = rng.choice(N_CLASSES, size=len(y_val), p=prior)
record(evaluate_predictions(
    y_val, stratified_pred, np.tile(prior, (len(y_val), 1)), "Baseline: stratified random"
))

## 3. Model 1: Engineered Visual Features + Random Forest

The Random Forest tests whether simple colour, shape and coverage measurements are sufficient. HSV histograms describe palette and brightness, HOG describes edges and silhouette, and foreground occupancy describes how much of the frame the product covers.


In [ ]:
FEATURE_CONFIG = {
    "hue_bins": 12, "saturation_bins": 8, "value_bins": 8,
    "hog_orientations": 9, "hog_pixels_per_cell": (8, 8),
    "hog_cells_per_block": (2, 2), "foreground_threshold": 0.95,
}

def extract_visual_features(image_uint8):
    rgb = image_uint8.astype(np.float32) / 255.0
    hsv = rgb2hsv(rgb)
    gray = rgb2gray(rgb)
    values = []

    for image in (rgb, hsv):
        for channel in range(3):
            x = image[:, :, channel]
            values.extend([x.mean(), x.std(), np.percentile(x, 25),
                           np.percentile(x, 50), np.percentile(x, 75)])

    for channel, bins in enumerate((FEATURE_CONFIG["hue_bins"],
                                    FEATURE_CONFIG["saturation_bins"],
                                    FEATURE_CONFIG["value_bins"])):
        hist, _ = np.histogram(hsv[:, :, channel], bins=bins, range=(0, 1))
        values.extend(hist / max(hist.sum(), 1))

    values.extend(hog(gray, orientations=FEATURE_CONFIG["hog_orientations"],
                      pixels_per_cell=FEATURE_CONFIG["hog_pixels_per_cell"],
                      cells_per_block=FEATURE_CONFIG["hog_cells_per_block"],
                      block_norm="L2-Hys", feature_vector=True))

    foreground = np.any(rgb < FEATURE_CONFIG["foreground_threshold"], axis=2)
    h, w = foreground.shape
    values.extend([foreground.mean(), foreground[:h//2].mean(), foreground[h//2:].mean(),
                   foreground[:, :w//2].mean(), foreground[:, w//2:].mean()])
    return np.asarray(values, dtype=np.float32)


def visual_feature_matrix(images, description):
    start = time.time()
    rows = Parallel(n_jobs=FEATURE_N_JOBS)(
        delayed(extract_visual_features)(image) for image in images
    )
    matrix = np.vstack(rows)
    print(f"{description}: {matrix.shape} in {time.time() - start:.1f}s")
    assert np.isfinite(matrix).all()
    return matrix


FEATURE_FINGERPRINT = hashlib.sha1(json.dumps({
    "config": FEATURE_CONFIG, "train_ids": train_frame["id"].astype(str).tolist(),
    "validation_ids": val_frame["id"].astype(str).tolist(),
}, sort_keys=True).encode()).hexdigest()[:12]
FEATURE_CACHE = CHECKPOINT_DIR / f"random_forest_features_{FEATURE_FINGERPRINT}.npz"
if RESUME and FEATURE_CACHE.exists():
    cached = np.load(FEATURE_CACHE)
    X_train_features, X_val_features = cached["train"], cached["validation"]
    print("Restored engineered features from", FEATURE_CACHE.name)
else:
    X_train_features = visual_feature_matrix(X_train_images, "training features")
    X_val_features = visual_feature_matrix(X_val_images, "validation features")
    np.savez_compressed(FEATURE_CACHE, train=X_train_features, validation=X_val_features)

print("Features per image:", X_train_features.shape[1])


### 3.1 Random Forest training

The forest models nonlinear interactions between the engineered features. Balanced class weights reduce bias toward the largest season without changing the validation distribution.


In [ ]:
RF_PATH = CHECKPOINT_DIR / f"random_forest_{FEATURE_FINGERPRINT}.joblib"
if RESUME and RF_PATH.exists():
    random_forest = joblib_load(RF_PATH)
    print("Restored", RF_PATH.name)
else:
    random_forest = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF, max_features=RF_MAX_FEATURES,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    )
    random_forest.fit(X_train_features, y_train)
    joblib_dump(random_forest, RF_PATH)

rf_scores = random_forest.predict_proba(X_val_features)
rf_pred = random_forest.classes_[rf_scores.argmax(axis=1)]
record(evaluate_predictions(y_val, rf_pred, rf_scores, "1. Engineered features + Random Forest"))
display(per_class_table(y_val, rf_pred).style.format({
    "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))


### 3.2 Feature evidence

Permutation importance measures how validation macro-F1 changes when a feature is shuffled. It is descriptive evidence, not a causal explanation.


In [ ]:
importance = permutation_importance(
    random_forest, X_val_features, y_val, scoring="f1_macro",
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
top = np.argsort(importance.importances_mean)[-20:]
plt.figure(figsize=(8, 5))
plt.barh(range(len(top)), importance.importances_mean[top], color=PALETTE[0])
plt.yticks(range(len(top)), [f"feature {i}" for i in top])
plt.xlabel("Decrease in validation macro-F1")
plt.title("Random Forest permutation importance: top 20 features")
plt.tight_layout(); plt.show()


## 4. Shared Neural-Network Training and Recovery

EfficientNet-B0 and DenseNet-121 use the same optimiser, schedule, early-stopping rule and validation function. Training stops on validation macro-F1, and the best epoch is restored. Full checkpoints preserve the optimiser, scheduler, mixed-precision scaler and random-number state.


In [ ]:
# --- Run identity ------------------------------------------------------------------------
# A checkpoint is only safe to reuse if it was produced by the same problem and the same recipe.
# The fingerprint below is stored inside every checkpoint and compared on load, so changing a
# hyperparameter silently invalidates the old files instead of silently reusing them.
RUN_FINGERPRINT = hashlib.sha1(json.dumps({
    "target": TARGET, "classes": N_CLASSES, "train_rows": int(len(y_train)),
    "val_rows": int(len(y_val)), "seed": RANDOM_STATE, "quick": QUICK_RUN,
    "validation_share": VALIDATION_SHARE, "batch": BATCH_SIZE, "epochs": EPOCHS,
    "patience": PATIENCE, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "warmup": WARMUP_EPOCHS, "label_smoothing": LABEL_SMOOTHING,
    "aug": [AUG_FLIP_PROBABILITY, AUG_ROTATION_DEGREES, AUG_TRANSLATE_FRACTION,
            AUG_JITTER_STRENGTH],
    "models": ["efficientnet_b0", "densenet121"], "pretrained": USE_PRETRAINED,
    "norm": [NORM_MEAN.round(5).tolist(), NORM_STD.round(5).tolist()],
}, sort_keys=True).encode()).hexdigest()[:12]
print("Run fingerprint:", RUN_FINGERPRINT)


def slug(text):
    """A filesystem-safe key for a run label."""
    return "".join(c if c.isalnum() else "_" for c in text.lower()).strip("_")


def checkpoint_path(name, kind="model"):
    return CHECKPOINT_DIR / f"{kind}_{slug(name)}.pt"


def torch_load(path):
    """Load a checkpoint. weights_only=False because these blobs carry history and RNG state."""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:                                    # torch < 2.0 has no weights_only
        return torch.load(path, map_location="cpu")


def atomic_save(payload, path):
    """Write through a temporary file so an interrupted save cannot corrupt the checkpoint."""
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def model_state(model):
    """CPU copy of a state dict, with any torch.compile prefix stripped.

    Stripping `_orig_mod.` is what lets a checkpoint written by a compiled run load into an
    eager model and the reverse, so USE_COMPILE can be toggled without orphaning the files.
    """
    return {key.replace("_orig_mod.", ""): value.detach().cpu().clone()
            for key, value in model.state_dict().items()}


def load_model_state(model, state):
    """Load a stripped state dict into a model that may or may not be compiled."""
    if any(key.startswith("_orig_mod.") for key in model.state_dict()):
        state = {f"_orig_mod.{key}": value for key, value in state.items()}
    model.load_state_dict(state)


def prepare_model(model, compile_ok=True):
    """Move a model to the device in the memory layout the convolution kernels prefer."""
    model = model.to(DEVICE)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    if USE_COMPILE and compile_ok:
        try:
            model = torch.compile(model)
        except Exception as error:                       # noqa: BLE001 - any failure is a fallback
            print(f"  torch.compile unavailable, continuing eagerly: {error}")
    return model


def run_epoch(model, loader, criterion, optimiser=None, logit_bias=None, scaler=None):
    """One pass over a stream. Trains if an optimiser is given, otherwise evaluates.

    Args:
        logit_bias: optional (N_CLASSES,) tensor added to the logits before the loss.
            Used by logit-adjusted cross-entropy in Section 6; never applied at evaluation,
            because the adjustment belongs to the training objective and not to the model.
        scaler: gradient scaler, required only for fp16. bf16 needs none.

    Returns:
        (mean loss, logits array or None). Logits are returned in float32 whatever the
        autocast dtype was, so every metric downstream sees the same precision as before.

    The running loss and the collected logits both stay on the device until the pass ends.
    Calling `.item()` per batch, as the original did, forces a host synchronisation on every
    step, and at this model size the step is short enough for that stall to be a real share of
    the epoch. One synchronisation per epoch reports the identical number.
    """
    training = optimiser is not None
    model.train(training)
    loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
    n_seen, collected = 0, []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits if logit_bias is None else logits + logit_bias, labels)

            if training:
                optimiser.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimiser)
                    scaler.update()
                else:
                    loss.backward()
                    optimiser.step()
            else:
                collected.append(logits.detach().float())

            loss_sum += loss.detach().float() * labels.shape[0]
            n_seen += labels.shape[0]

    logits_out = torch.cat(collected).cpu().numpy() if collected else None
    return (loss_sum / max(n_seen, 1)).item(), logits_out


def make_scaler():
    """A gradient scaler, enabled only for fp16. bf16 has fp32's range and needs no scaling."""
    enabled = AMP_ENABLED and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler(DEVICE.type, enabled=enabled)
    except (AttributeError, TypeError):      # torch < 2.4 keeps it under torch.cuda.amp
        return torch.cuda.amp.GradScaler(enabled=enabled)


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, logit_bias=None, patience=PATIENCE,
        parameters=None, label="", verbose_every=1, run_name=None):
    """Train with warmup + cosine decay, early stopping on validation macro-F1.

    Mid-run checkpointing writes the model, the optimiser, the scheduler, the gradient scaler,
    the epoch counter, the best-so-far state and every random generator's state. That full set
    is what makes a resumed run the same run rather than a similar one: restoring the weights
    alone would restart Adam's moments from zero and redraw a different augmentation stream,
    both of which change the result.

    Returns:
        (history DataFrame, best validation logits, best macro-F1).
    """
    run_name = run_name or label
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimiser = torch.optim.AdamW(
        parameters if parameters is not None else model.parameters(),
        lr=lr, weight_decay=weight_decay,
    )
    scaler = make_scaler()

    def schedule(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / max(WARMUP_EPOCHS, 1)
        progress = (epoch - WARMUP_EPOCHS) / max(epochs - WARMUP_EPOCHS, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, schedule)

    history, best = [], {"macro_f1": -1.0, "epoch": -1, "state": None, "logits": None}
    first_epoch = 0
    resume_path = checkpoint_path(run_name, "epoch")

    if RESUME and resume_path.exists():
        blob = torch_load(resume_path)
        if blob.get("fingerprint") == RUN_FINGERPRINT:
            load_model_state(model, blob["model"])
            optimiser.load_state_dict(blob["optimiser"])
            scheduler.load_state_dict(blob["scheduler"])
            if blob.get("scaler") is not None:
                scaler.load_state_dict(blob["scaler"])
            restore_rng_state(blob["rng"])
            history, best, first_epoch = blob["history"], blob["best"], blob["epoch"]
            print(f"  [{label}] resumed at epoch {first_epoch + 1}/{epochs}; "
                  f"best so far {best['macro_f1']:.4f} at epoch {best['epoch']}")
        else:
            print(f"  [{label}] epoch checkpoint ignored: the configuration changed since it "
                  "was written")

    start = time.time()

    for epoch in range(first_epoch, epochs):
        epoch_start = time.time()
        train_loss, _ = run_epoch(model, train_loader, criterion, optimiser, logit_bias, scaler)
        val_loss, val_logits = run_epoch(model, val_loader, criterion, None, None, None)
        scheduler.step()

        val_pred = val_logits.argmax(axis=1)
        macro = f1_score(y_val, val_pred, labels=SCOREABLE, average="macro", zero_division=0)
        history.append({
            "epoch": epoch + 1, "train loss": train_loss, "val loss": val_loss,
            "val accuracy": accuracy_score(y_val, val_pred), "val macro-F1": macro,
            "lr": optimiser.param_groups[0]["lr"], "seconds": time.time() - epoch_start,
        })

        improved = macro > best["macro_f1"]
        if improved:
            best.update({
                "macro_f1": macro, "epoch": epoch + 1,
                "state": model_state(model),
                "logits": val_logits,
            })

        # Every epoch by default. Silence across several minutes is indistinguishable from a
        # hang, which is what made the original five-epoch reporting interval a poor default.
        if verbose_every and ((epoch + 1) % verbose_every == 0 or epoch == 0):
            print(f"  [{label}] epoch {epoch + 1:>3}/{epochs}  train {train_loss:.3f}  "
                  f"val {val_loss:.3f}  acc {history[-1]['val accuracy']:.3f}  "
                  f"macro-F1 {macro:.4f}  {history[-1]['seconds']:.0f}s")

        stopping = epoch + 1 - best["epoch"] >= patience
        # Not on the stopping epoch: the run is about to finish, `bank` writes the finished
        # model, and this file is deleted immediately below. Writing it would be a wasted
        # save of the whole optimiser state.
        if not stopping and (improved or (epoch + 1) % max(CHECKPOINT_EVERY, 1) == 0):
            atomic_save({
                "fingerprint": RUN_FINGERPRINT, "epoch": epoch + 1,
                "model": model_state(model),
                "optimiser": optimiser.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler.is_enabled() else None,
                "rng": capture_rng_state(), "history": history, "best": best,
            }, resume_path)

        if stopping:
            print(f"  [{label}] early stop at epoch {epoch + 1}; best was epoch {best['epoch']}")
            break

    if best["state"] is None:
        raise RuntimeError(f"[{label}] finished without a scored epoch.")

    load_model_state(model, best["state"])   # report and save the same weights
    print(f"  [{label}] best macro-F1 {best['macro_f1']:.4f} at epoch {best['epoch']} "
          f"({time.time() - start:.0f}s this session, "
          f"{np.mean([h['seconds'] for h in history]):.0f}s per epoch)")

    if resume_path.exists() and not KEEP_EPOCH_CHECKPOINTS:
        resume_path.unlink()      # the finished model is banked below; the mid-run file is spent

    return pd.DataFrame(history), best["logits"], best["macro_f1"]


def bank(name, model, logits, history=None, extra=None):
    """Write a finished model to disk as soon as it exists.

    Section 8 is the last training step in the notebook, which means a run interrupted anywhere before
    it would lose every model it had already trained. Banking here costs a second and makes each
    model independently recoverable, and `restore` below is what turns that into a resumed run
    rather than merely a backup.
    """
    payload = {
        "name": name, "fingerprint": RUN_FINGERPRINT,
        "state_dict": model_state(model),
        "val_logits": logits,
        "history": None if history is None else history.to_dict("records"),
        "classes": CLASSES,
        "normalisation_mean": NORM_MEAN.tolist(),
        "normalisation_std": NORM_STD.tolist(),
        "image_target_size": list(IMAGE_TARGET_SIZE),
    }
    if extra:
        payload.update(extra)
    path = checkpoint_path(name, "model")
    atomic_save(payload, path)
    print(f"  banked -> {path.name} ({path.stat().st_size / 1e6:.1f} MB)")


def restore(name, model=None):
    """Load a banked model if one exists for this exact configuration, else return None.

    The fingerprint check is the safety property. A checkpoint written under a different
    learning rate or a different split is not a shortcut, it is a wrong answer, so it is
    ignored rather than loaded.
    """
    if not RESUME:
        return None
    path = checkpoint_path(name, "model")
    if not path.exists():
        return None
    blob = torch_load(path)
    if blob.get("fingerprint") != RUN_FINGERPRINT:
        print(f"  '{name}': banked checkpoint ignored, the configuration changed since it "
              "was written")
        return None
    if model is not None:
        load_model_state(model, blob["state_dict"])
    records = blob.get("history")
    blob["history"] = pd.DataFrame(records) if records else None
    print(f"  '{name}': restored from {path.name}, training skipped")
    return blob


def train_or_restore(name, build, label, compile_ok=True, **fit_kwargs):
    """Return a trained model, restoring it from disk when this configuration already produced it.

    The restored path and the trained path return the same three objects, so every cell below
    reads identically whether the model was just trained or recovered from a previous session.
    """
    model = prepare_model(build(), compile_ok=compile_ok)
    cached = restore(name, model)
    if cached is not None:
        return model, cached["history"], cached["val_logits"]
    history, logits, _ = fit(model, train_loader, val_loader, label=label, run_name=name,
                             **fit_kwargs)
    bank(name, model, logits, history)
    return model, history, logits


def release(*names):
    """Drop models the notebook no longer reads and return their VRAM."""
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def plot_history(histories, title):
    """Training curves for one or more runs."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for (label, history), colour in zip(histories.items(), PALETTE):
        if history is None:
            continue
        axes[0].plot(history["epoch"], history["train loss"], color=colour, label=f"{label} train")
        axes[0].plot(history["epoch"], history["val loss"], color=colour, ls="--",
                     label=f"{label} val")
        axes[1].plot(history["epoch"], history["val macro-F1"], color=colour, label=label)
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
    axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
    for ax in axes:
        ax.legend(fontsize=8)
    fig.suptitle(title, y=1.03)
    plt.tight_layout(); plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Model 2: EfficientNet-B0

EfficientNet-B0 learns colour, texture and garment structure directly from images. Its MBConv blocks reduce computation, while squeeze-and-excitation recalibrates channel responses. It tests whether an efficient learned representation improves on engineered features.


In [ ]:
def build_efficientnet():
    if USE_PRETRAINED:
        raise ValueError("Pretrained weights are disabled in this from-scratch notebook.")
    model = efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, N_CLASSES)
    return model

set_seed(RANDOM_STATE)
efficientnet, efficientnet_history, efficientnet_logits = train_or_restore(
    "efficientnet_b0", build_efficientnet, label="EfficientNet-B0")
efficientnet_pred = efficientnet_logits.argmax(axis=1)
record(evaluate_predictions(y_val, efficientnet_pred, efficientnet_logits,
                            "2. EfficientNet-B0"))
plot_history({"EfficientNet-B0": efficientnet_history}, "EfficientNet-B0 training")


## 6. Model 3: DenseNet-121

DenseNet-121 concatenates earlier feature maps into later layers. This feature reuse keeps low-level colour, edge and texture information available while higher-level garment features are learned. It tests whether dense reuse is more useful than EfficientNet's efficiency-focused design.


In [ ]:
def build_densenet():
    if USE_PRETRAINED:
        raise ValueError("Pretrained weights are disabled in this from-scratch notebook.")
    model = densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, N_CLASSES)
    return model

set_seed(RANDOM_STATE)
densenet, densenet_history, densenet_logits = train_or_restore(
    "densenet121", build_densenet, label="DenseNet-121")
densenet_pred = densenet_logits.argmax(axis=1)
record(evaluate_predictions(y_val, densenet_pred, densenet_logits,
                            "3. DenseNet-121"))
plot_history({"DenseNet-121": densenet_history}, "DenseNet-121 training")


## 7. Ultimate Judgement

The metric table selects a leading model, then the following checks examine where it succeeds, how it fails and what it costs. The automatically selected leader should still be reviewed before the report recommendation is written.


In [ ]:
summary_table = pd.concat(RESULTS, ignore_index=True)
display(summary_table.style.format({c: "{:.4f}" for c in summary_table.columns if c != "Model"})
        .background_gradient(subset=["Macro-F1"], cmap="Blues"))

candidate_rows = summary_table[summary_table["Model"].str.match(r"^[123]\.")]
FINAL_NAME = candidate_rows.sort_values("Macro-F1", ascending=False).iloc[0]["Model"]
print("Metric-table leader:", FINAL_NAME)

if "Random Forest" in FINAL_NAME:
    FINAL_KIND, final_model, final_logits, final_pred = "random_forest", random_forest, rf_scores, rf_pred
elif "EfficientNet" in FINAL_NAME:
    FINAL_KIND, final_model, final_logits, final_pred = "neural", efficientnet, efficientnet_logits, efficientnet_pred
else:
    FINAL_KIND, final_model, final_logits, final_pred = "neural", densenet, densenet_logits, densenet_pred


### 7.1 Per-season performance and confusion


In [ ]:
display(per_class_table(y_val, final_pred).style.format({
    "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))

confusion = confusion_matrix(y_val, final_pred, labels=np.arange(N_CLASSES), normalize="true")
plt.figure(figsize=(7, 6))
sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, vmin=0, vmax=1)
plt.title(f"{FINAL_NAME}: row-normalised confusion matrix")
plt.xlabel("Predicted season"); plt.ylabel("True season")
plt.tight_layout(); plt.show()


### 7.2 Calibration


In [ ]:
def expected_calibration_error(probabilities, y_true, n_bins=10):
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = predicted == y_true
    edges = np.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (confidence > low) & (confidence <= high)
        if mask.any():
            accuracy, mean_confidence = correct[mask].mean(), confidence[mask].mean()
            ece += mask.mean() * abs(accuracy - mean_confidence)
            rows.append((mean_confidence, accuracy, mask.mean()))
    return ece, pd.DataFrame(rows, columns=["Confidence", "Accuracy", "Share"])


if FINAL_KIND == "neural":
    final_probabilities = torch.softmax(torch.from_numpy(final_logits), dim=1).numpy()
else:
    final_probabilities = final_logits

ece, reliability = expected_calibration_error(final_probabilities, y_val)
print(f"Expected Calibration Error: {ece:.4f}")
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color=MUTED, label="Perfect calibration")
plt.plot(reliability["Confidence"], reliability["Accuracy"], "o-", label=FINAL_NAME)
plt.xlabel("Mean confidence"); plt.ylabel("Accuracy"); plt.legend()
plt.tight_layout(); plt.show()


### 7.3 Deployment cost and model attention


In [ ]:
if FINAL_KIND == "neural":
    print("Trainable parameters:", f"{count_parameters(final_model):,}")
    if RUN_SALIENCY:
        final_model.eval()
        chosen = np.random.RandomState(RANDOM_STATE).choice(len(y_val), min(6, len(y_val)), replace=False)
        index = torch.as_tensor(chosen, device=val_loader.images.device)
        raw = val_loader.images[index].to(DEVICE).permute(0, 3, 1, 2).float().div(255.0)
        normalised = (raw - NORM_MEAN_T) / NORM_STD_T
        fig, axes = plt.subplots(2, len(chosen), figsize=(2.2 * len(chosen), 4.5))
        for column, position in enumerate(chosen):
            image = normalised[column:column+1].clone().requires_grad_(True)
            logits = final_model(image)
            logits[0, logits.argmax()].backward()
            saliency = image.grad.abs().max(dim=1)[0].squeeze().detach().cpu()
            axes[0, column].imshow(raw[column].permute(1, 2, 0).cpu())
            axes[1, column].imshow(saliency, cmap="inferno")
            axes[0, column].set_title(f"true {CLASSES[y_val[position]]}\npred {CLASSES[final_pred[position]]}", fontsize=8)
            axes[0, column].axis("off"); axes[1, column].axis("off")
        plt.tight_layout(); plt.show()
else:
    print("Random Forest file size (MB):", RF_PATH.stat().st_size / 1e6)
    print("Saliency is only defined here for neural models; use permutation importance for the forest.")


### 7.4 The judgement

Write the final recommendation using the evidence above. State the macro-F1 margin, per-season weaknesses, calibration and deployment cost. Do not select a model from ImageNet accuracy or architecture reputation alone.


## 8. Persisting the Model and Producing Predictions

The saved artifacts include the class mapping, normalisation constants, feature configuration and run settings. Test images use the same deterministic transform and training-fitted statistics.


In [ ]:
summary_table.to_csv(ARTEFACT_DIR / "task2_results.csv", index=False)
(ARTEFACT_DIR / "task2_classes.json").write_text(json.dumps(CLASSES, indent=2))

metadata = {
    "model_name": FINAL_NAME, "model_kind": FINAL_KIND,
    "classes": CLASSES, "normalisation_mean": NORM_MEAN.tolist(),
    "normalisation_std": NORM_STD.tolist(), "image_target_size": list(IMAGE_TARGET_SIZE),
    "feature_config": FEATURE_CONFIG,
}
(ARTEFACT_DIR / "task2_config.json").write_text(json.dumps(metadata, indent=2))

if FINAL_KIND == "neural":
    atomic_save({**metadata, "state_dict": model_state(final_model)},
                ARTEFACT_DIR / "task2_model.pt")
else:
    joblib_dump(final_model, ARTEFACT_DIR / "task2_random_forest.joblib")

template = pd.read_csv(PREDICTION_TEMPLATE)
test_paths = [str(Path(TEST_IMAGE_DIR) / f"{image_id}.jpg") for image_id in template["id"]]
missing = [path for path in test_paths if not Path(path).exists()]
assert not missing, f"{len(missing)} test images are missing"

width, height = IMAGE_TARGET_SIZE
X_test_images = np.empty((len(test_paths), height, width, 3), dtype=np.uint8)
for position, path in enumerate(test_paths):
    X_test_images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)

if FINAL_KIND == "random_forest":
    X_test_features = visual_feature_matrix(X_test_images, "test features")
    test_scores = final_model.predict_proba(X_test_features)
    test_indices = final_model.classes_[test_scores.argmax(axis=1)]
else:
    test_loader = BatchStream(X_test_images, np.zeros(len(X_test_images), dtype=np.int64),
                              batch_size=512, augment=False, shuffle=False)
    final_model.eval()
    chunks = []
    with torch.no_grad():
        for images, _ in test_loader:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                chunks.append(final_model(images).float().cpu())
    test_scores = torch.cat(chunks).numpy()
    test_indices = test_scores.argmax(axis=1)

predictions = template.copy()
predictions[TARGET] = [CLASSES[index] for index in test_indices]
assert predictions[TARGET].notna().all() and len(predictions) == len(template)
prediction_path = OUTPUT_DIR / "task2_predictions.csv"
predictions.to_csv(prediction_path, index=False)
np.save(OUTPUT_DIR / "task2_test_scores.npy", test_scores.astype(np.float32))
print("Written:", prediction_path.resolve())
display(predictions.head())


## 9. Decision Log, Limitations, and What to Tune Next

### 9.1 Decision log

| Decision | Reason |
|---|---|
| Macro-F1 is primary | It gives every season equal weight. |
| Random Forest uses engineered features | Trees do not model image geometry well from flattened pixels. |
| Colour jitter is mild | Colour may contain valid season information. |
| EfficientNet and DenseNet share one training loop | This makes their comparison fair. |
| Top-2 replaces top-5 | The season label space is small. |

### 9.2 Limitations

- Season labels may be subjective or assigned for commercial reasons that are not visible in the image.
- Some products are suitable for several seasons, but the task provides one label.
- Colour relationships may reflect this catalogue rather than a general fashion rule.
- Native 60x80 images contain limited fabric and texture detail.
- Saliency is a qualitative check and does not prove what caused a prediction.

### 9.3 Tuning order

1. Check learning rate for both neural models.
2. Compare no colour jitter against mild colour jitter.
3. Tune Random Forest depth and minimum leaf size.
4. Tune weight decay, dropout and batch size.

### 9.4 Further work

- Test feature-group ablations for colour, HOG and foreground occupancy.
- Apply temperature scaling if the selected model is poorly calibrated.
- Compare transfer learning only if the assignment permits pretrained weights.
